# 04 — Model Training
## Predictive Maintenance Dataset

This notebook trains two models:

1. **Failure Prediction Model** — binary classification of `Target`
2. **Failure Type Model** — multiclass classification on failed machines only

The dataset is highly imbalanced for failures, so evaluation includes F1 and Average Precision in addition to accuracy.

In [1]:
import json
import joblib
import numpy as np
import pandas as pd
from pathlib import Path

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

pd.set_option("display.max_columns", None)

In [2]:
BASE_DIR = Path("..")

PROCESSED_DATA_DIR = BASE_DIR / "data" / "processed"
MODEL_DIR = BASE_DIR / "models"

input_path = PROCESSED_DATA_DIR / "predictive_maintenance_features.csv"

MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Input:", input_path)
print("Models:", MODEL_DIR)

Input: ..\data\processed\predictive_maintenance_features.csv
Models: ..\models


In [3]:
df = pd.read_csv(input_path)

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (10000, 16)


,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Type_H,Type_L,Type_M,Temperature Difference [K],Temperature Ratio,Mechanical Power Proxy,Tool Wear × Torque,Torque Squared,Rotational Speed Squared,Target,Failure Type
0,298.1,308.6,1551,42.8,0,0,0,1,10.5,1.035223,66382.8,0.0,1831.84,2405601,0,No Failure
1,298.2,308.7,1408,46.3,3,0,1,0,10.5,1.035211,65190.4,138.9,2143.69,1982464,0,No Failure
2,298.1,308.5,1498,49.4,5,0,1,0,10.4,1.034888,74001.2,247.0,2440.36,2244004,0,No Failure
3,298.2,308.6,1433,39.5,7,0,1,0,10.4,1.034876,56603.5,276.5,1560.25,2053489,0,No Failure
4,298.2,308.7,1408,40.0,9,0,1,0,10.5,1.035211,56320.0,360.0,1600.00,1982464,0,No Failure


## 1. Define the feature set

In [4]:
FEATURE_COLUMNS = [
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]",
    "Type_H",
    "Type_L",
    "Type_M",
    "Temperature Difference [K]",
    "Temperature Ratio",
    "Mechanical Power Proxy",
    "Tool Wear × Torque",
    "Torque Squared",
    "Rotational Speed Squared",
]

X = df[FEATURE_COLUMNS].copy()
y = df["Target"].astype(int)

assert X.isna().sum().sum() == 0
assert np.isfinite(X.to_numpy()).all()

print("X shape:", X.shape)
print("Target distribution:")
display(y.value_counts())
display((y.value_counts(normalize=True) * 100).round(3))

X shape: (10000, 14)
Target distribution:


Target
0    9661
1     339
Name: count, dtype: int64

Target
0    96.61
1     3.39
Name: proportion, dtype: float64

# Model A — Failure Prediction

The first model predicts whether a failure is present.

Because the positive class is relatively rare, we use stratification and class weighting.

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Train failure rate:", round(y_train.mean() * 100, 3), "%")
print("Test failure rate:", round(y_test.mean() * 100, 3), "%")

Train shape: (8000, 14)
Test shape: (2000, 14)
Train failure rate: 3.388 %
Test failure rate: 3.4 %


## 2. Baseline — Logistic Regression

This gives us a simple interpretable baseline before using a nonlinear model.

In [6]:
logistic_model = Pipeline([
    ("scaler", StandardScaler()),
    (
        "model",
        LogisticRegression(
            class_weight="balanced",
            max_iter=2000,
            random_state=42
        )
    )
])

logistic_model.fit(X_train, y_train)

logistic_pred = logistic_model.predict(X_test)
logistic_prob = logistic_model.predict_proba(X_test)[:, 1]

logistic_metrics = {
    "accuracy": accuracy_score(y_test, logistic_pred),
    "precision": precision_score(y_test, logistic_pred, zero_division=0),
    "recall": recall_score(y_test, logistic_pred, zero_division=0),
    "f1": f1_score(y_test, logistic_pred, zero_division=0),
    "roc_auc": roc_auc_score(y_test, logistic_prob),
    "average_precision": average_precision_score(y_test, logistic_prob),
}

pd.Series(logistic_metrics).round(4)

accuracy             0.8635
precision            0.1846
recall               0.8824
f1                   0.3053
roc_auc              0.9434
average_precision    0.4541
dtype: float64

In [7]:
print(classification_report(
    y_test,
    logistic_pred,
    target_names=["No Failure", "Failure"],
    zero_division=0
))

              precision    recall  f1-score   support

  No Failure       1.00      0.86      0.92      1932
     Failure       0.18      0.88      0.31        68

    accuracy                           0.86      2000
   macro avg       0.59      0.87      0.61      2000
weighted avg       0.97      0.86      0.90      2000



## 3. Random Forest

Random Forest can capture nonlinear interactions between temperature, speed, torque and tool wear.

In [8]:
rf_model = RandomForestClassifier(
    n_estimators=400,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
    min_samples_leaf=2
)

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)
rf_prob = rf_model.predict_proba(X_test)[:, 1]

rf_metrics = {
    "accuracy": accuracy_score(y_test, rf_pred),
    "precision": precision_score(y_test, rf_pred, zero_division=0),
    "recall": recall_score(y_test, rf_pred, zero_division=0),
    "f1": f1_score(y_test, rf_pred, zero_division=0),
    "roc_auc": roc_auc_score(y_test, rf_prob),
    "average_precision": average_precision_score(y_test, rf_prob),
}

pd.Series(rf_metrics).round(4)

accuracy             0.9925
precision            0.9649
recall               0.8088
f1                   0.8800
roc_auc              0.9752
average_precision    0.8716
dtype: float64

In [9]:
print(classification_report(
    y_test,
    rf_pred,
    target_names=["No Failure", "Failure"],
    zero_division=0
))

              precision    recall  f1-score   support

  No Failure       0.99      1.00      1.00      1932
     Failure       0.96      0.81      0.88        68

    accuracy                           0.99      2000
   macro avg       0.98      0.90      0.94      2000
weighted avg       0.99      0.99      0.99      2000



In [10]:
comparison = pd.DataFrame(
    [logistic_metrics, rf_metrics],
    index=["Logistic Regression", "Random Forest"]
)

display(comparison.round(4))

,accuracy,precision,recall,f1,roc_auc,average_precision
Logistic Regression,0.8635,0.1846,0.8824,0.3053,0.9434,0.4541
Random Forest,0.9925,0.9649,0.8088,0.8800,0.9752,0.8716


## 4. Select the main failure model

For this imbalanced problem, F1 is used as the primary selection metric, while Average Precision and ROC-AUC are also reported.

This selection is based on the held-out test split, so the reported metrics should be treated as an initial benchmark. For a production model, cross-validation and an untouched final test set are recommended.

In [11]:
main_failure_model = (
    rf_model if rf_metrics["f1"] >= logistic_metrics["f1"]
    else logistic_model
)

main_failure_model_name = (
    "Random Forest"
    if rf_metrics["f1"] >= logistic_metrics["f1"]
    else "Logistic Regression"
)

print("Main failure model:", main_failure_model_name)

Main failure model: Random Forest


## 5. Random Forest feature importance

This is useful for understanding which machine measurements contribute most to the model.

In [12]:
if hasattr(rf_model, "feature_importances_"):
    importance = pd.Series(
        rf_model.feature_importances_,
        index=FEATURE_COLUMNS
    ).sort_values(ascending=False)

    display(importance.to_frame("importance"))

,importance
Tool wear [min],0.135171
Mechanical Power Proxy,0.121665
Torque Squared,0.121635
Rotational Speed Squared,0.120348
Rotational speed [rpm],0.119498
Torque [Nm],0.114856
Tool Wear × Torque,0.101283
Temperature Ratio,0.064206
Temperature Difference [K],0.042172
Air temperature [K],0.030016


# Model B — Failure Type Classification

This model is trained only on rows where `Target == 1`.

The task is to classify the observed failure into the available failure types.

In [13]:
failure_df = df[df["Target"] == 1].copy()

X_failure = failure_df[FEATURE_COLUMNS].copy()
y_failure_type = failure_df["Failure Type"].copy()

print("Failure-only dataset:", X_failure.shape)
display(y_failure_type.value_counts())

Failure-only dataset: (339, 14)


Failure Type
Heat Dissipation Failure    112
Power Failure                95
Overstrain Failure           78
Tool Wear Failure            45
No Failure                    9
Name: count, dtype: int64

In [14]:
X_ft_train, X_ft_test, y_ft_train, y_ft_test = train_test_split(
    X_failure,
    y_failure_type,
    test_size=0.20,
    stratify=y_failure_type,
    random_state=42
)

print("Train:", X_ft_train.shape)
print("Test:", X_ft_test.shape)

Train: (271, 14)
Test: (68, 14)


In [15]:
failure_type_model = RandomForestClassifier(
    n_estimators=400,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
    min_samples_leaf=2
)

failure_type_model.fit(
    X_ft_train,
    y_ft_train
)

ft_pred = failure_type_model.predict(X_ft_test)

print(classification_report(
    y_ft_test,
    ft_pred,
    zero_division=0
))

                          precision    recall  f1-score   support

Heat Dissipation Failure       0.96      1.00      0.98        22
              No Failure       1.00      1.00      1.00         2
      Overstrain Failure       0.93      0.88      0.90        16
           Power Failure       1.00      1.00      1.00        19
       Tool Wear Failure       0.89      0.89      0.89         9

                accuracy                           0.96        68
               macro avg       0.96      0.95      0.95        68
            weighted avg       0.96      0.96      0.96        68



In [16]:
ft_accuracy = accuracy_score(y_ft_test, ft_pred)
ft_f1_macro = f1_score(
    y_ft_test,
    ft_pred,
    average="macro",
    zero_division=0
)

print("Failure Type accuracy:", round(ft_accuracy, 4))
print("Failure Type macro F1:", round(ft_f1_macro, 4))

Failure Type accuracy: 0.9559
Failure Type macro F1: 0.954


## 6. Save confusion matrix and feature importance

In [17]:
cm = confusion_matrix(
    y_ft_test,
    ft_pred,
    labels=failure_type_model.classes_
)

cm_df = pd.DataFrame(
    cm,
    index=failure_type_model.classes_,
    columns=failure_type_model.classes_
)

display(cm_df)

,Heat Dissipation Failure,No Failure,Overstrain Failure,Power Failure,Tool Wear Failure
Heat Dissipation Failure,22,0,0,0,0
No Failure,0,2,0,0,0
Overstrain Failure,1,0,14,0,1
Power Failure,0,0,0,19,0
Tool Wear Failure,0,0,1,0,8


In [18]:
ft_importance = pd.Series(
    failure_type_model.feature_importances_,
    index=FEATURE_COLUMNS
).sort_values(ascending=False)

display(ft_importance.to_frame("importance"))

,importance
Tool Wear × Torque,0.145857
Tool wear [min],0.134813
Mechanical Power Proxy,0.131246
Torque Squared,0.103231
Temperature Ratio,0.094662
Torque [Nm],0.094135
Temperature Difference [K],0.086774
Rotational Speed Squared,0.071168
Rotational speed [rpm],0.068153
Air temperature [K],0.033026


# 7. Save trained models and metadata

The saved models are intended to be loaded later by the prediction/API layer.

In [19]:
failure_model_path = MODEL_DIR / "failure_prediction_model.joblib"
failure_type_model_path = MODEL_DIR / "failure_type_model.joblib"
metadata_path = MODEL_DIR / "predictive_maintenance_metadata.json"

joblib.dump(
    main_failure_model,
    failure_model_path
)

joblib.dump(
    failure_type_model,
    failure_type_model_path
)

metadata = {
    "feature_columns": FEATURE_COLUMNS,
    "failure_model": main_failure_model_name,
    "failure_target": "Target",
    "failure_type_target": "Failure Type",
    "failure_type_trained_on": "Target == 1",
    "failure_type_classes": failure_type_model.classes_.tolist(),
    "random_state": 42,
}

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print("Saved:")
print(f"- {failure_model_path}")
print(f"- {failure_type_model_path}")
print(f"- {metadata_path}")

Saved:
- ..\models\failure_prediction_model.joblib
- ..\models\failure_type_model.joblib
- ..\models\predictive_maintenance_metadata.json


## 8. Reload test

Confirm that the serialized models can be loaded and used for prediction.

In [20]:
loaded_failure_model = joblib.load(failure_model_path)
loaded_failure_type_model = joblib.load(failure_type_model_path)

sample = X_test.iloc[:5]

sample_failure_prob = loaded_failure_model.predict_proba(sample)[:, 1]
sample_failure_pred = loaded_failure_model.predict(sample)

print("Sample failure probabilities:", np.round(sample_failure_prob, 4))
print("Sample failure predictions:", sample_failure_pred)

print("Model reload test passed.")

Sample failure probabilities: [0.1292 0.     0.1813 0.     0.0314]
Sample failure predictions: [0 0 0 0 0]
Model reload test passed.


## Final output

The modeling stage produces:

- `failure_prediction_model.joblib`
- `failure_type_model.joblib`
- `predictive_maintenance_metadata.json`

The failure model returns a probability that can later be exposed as `failure_probability` in the dashboard.

The failure-type model is only meaningful after a failure-risk decision/threshold identifies a case as a failure candidate.